In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

class Neo4jHttpGraph:
    def __init__(self, query_url=None, username=None, password=None):
        self.query_url = query_url or f"https://{os.getenv('AURA_INSTANCEID')}.databases.neo4j.io/db/{os.getenv('NEO4J_DATABASE')}/query/v2"
        self.username = username or os.getenv("AURA_INSTANCEID")
        self.password = password or os.getenv("NEO4J_PASSWORD")
        self.auth = (self.username, self.password)
        # Connection test
        self.query("RETURN 1 AS test")
        print("Connected to Neo4j via HTTP Query API successfully!")

    def query(self, query: str, params: dict = None):
        payload = {"statement": query}
        if params:
            payload["parameters"] = params
        resp = requests.post(self.query_url, json=payload, auth=self.auth)
        resp.raise_for_status()
        data = resp.json().get("data", {})
        fields = data.get("fields", [])
        values = data.get("values", [])
        return [dict(zip(fields, row)) for row in values]

# Usage:
graph = Neo4jHttpGraph(
    query_url="https://b4b9f51e.databases.neo4j.io/db/b4b9f51e/query/v2",
    username=os.getenv("AURA_INSTANCEID"), 
    password=os.getenv("NEO4J_PASSWORD")
)

# Test query
results = graph.query("CALL dbms.components() YIELD name, versions RETURN name, versions")
print(results)



Connected to Neo4j via HTTP Query API successfully!
[{'name': 'Neo4j Kernel', 'versions': ['5.27-aura']}, {'name': 'Cypher', 'versions': ['5', '25']}]


In [3]:
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    groq_api_key=groq_api_key,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

In [4]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]

documents = [Document(page_content=text) for text in movie_texts]

documents


[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)

Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


In [6]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)


C:\Users\musta\AppData\Local\Temp\ipykernel_10304\3834549908.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [7]:
graph_documents

[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [8]:
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)


--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

In [9]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

print("Graph data added to Neo4j")


AttributeError: 'Neo4jHttpGraph' object has no attribute 'add_graph_documents'